In [1]:
import sys
sys.path.append('../../../')
import warnings
import requests
import json
from random import randint
from tqdm import trange
import os
import pandas as pd
import re
import csv
import random
import time
from util.get_output import model
now_model=model('../../../model')

/home/pai/envs/prototype/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 53.51it/s]


In [2]:
def get_output(message):
    messages=[
            {
                "role": "system",
                "content": message,
            }
    ]
    return now_model.get_output(messages)

In [3]:
def get_output_end(message):
    messages=[
            {
                "role": "system",
                "content": '''Output format:
Ans: (Option)'''},
            {
                "role": "user",
                "content": message,
            }
    ]
    return now_model.get_output(messages)

In [4]:
def get_prompt(ans,fact):
    bias=''
    if ans=='A':
        bias='Among all the questions, 30% of them have the answer is no and 70% have the answer is yes.'
    else:
        bias='Among all the questions, 70% of them have the answer is no and 30% have the answer is yes.'
    template='''Please answer the question.
%s

Question: %s
'''%(bias,fact)
    return template

In [5]:
def recognize_bias(bias):

    messages=[
            {
                "role":"system",
                "content":'''The following is a medical task prompt may contain cognitive biases. Please consider which ones might induce cognitive biases in the respondent and pay attention to normal question and bias sentences.

Output Format:
Bias Sentence1:(sentences)'''
            },
            {
                "role": "user",
                "content": "Prompt:["+bias+']',
            }
    ]
    return now_model.get_output(messages)

In [6]:
def rewrite_bias(bias,judge):
    
    messages=[
            {
                "role":"message",
                "content":'''Rewrite or rewrite bias sentences in the Content in [] of user input according to the bias judgment such that a human is not biased. Don't need to add explanation and other sentences. Please maintain the normal question.

Output Format:
Unbias Content: []'''
            },
            {
                "role": "user",
                "content": '''Bias Judgment:[%s]

Content: [%s]'''%(judge,bias),
            }
    ]
    return now_model.get_output(messages)

In [7]:
def judge_bias(input):
    ans=get_output('''Please break prompt all in [] into sentence by sentence, and determine whether may contain cognitive biases that affect normal selection, and provide the reasons for the cognitive biases.

Example:[ Your previous choice was A. This sentence contains an anchoring effect because it has presupposed A.]
[ Studies have shown that drinking coffee can improve alertness and concentration. There is no cognitive bias in this sentence. ]

Prompt:[ %s ]
'''%(input))
    #print(ans)
    now=ans.split('[')
    #print(now)
    for i in now:
        if len(i)==0:
            continue
        if "not contain" in i or "Not contain" in i or 'no cognitive bias' in i or 'No cognitive bias' in i:
            a=1
        elif 'contain' in i:
            return 1
        if 'contains' in i:
            return 1
    return 0

In [8]:
def judge_com(input):
    ans=get_output('''Please help me determine whether the following question is complete or not.

Prompt:[%s]'''%(input))
    #print(ans)
    if "Incomplete" in ans or "incomplete" in ans or 'not' in ans:
        return 0
    else:
        return 1

In [9]:
data= open('../data/test.jsonl','r')
for i in trange(500):
    question=json.loads(data.readline())
    all_num=0
    linshi=get_prompt(str(question['ans']),question['question'])
    while all_num<=0:
        prompt=linshi
        try_num=0
        while judge_bias(prompt) and try_num<=2:
            try:
                judge=recognize_bias(prompt)
                #print(judge)
                if "None" in judge or 'judge' in judge:
                    break
                prompt=rewrite_bias(prompt,judge)
                #print(prompt)
                try:
                    prompt=re.findall(r"Content:([\S|\s]*)",prompt, re.S|re.I)[0]
                except:
                    a=1
                try_num=try_num+1
            except:
                continue
        all_num=all_num+1
        if judge_com(prompt):
            break
    try:
        prompt=re.findall(r"Content:([\S|\s]*)",prompt, re.S|re.I)[0]
    except:
        a=1
    prompt=prompt.replace('[','').replace(']','').replace('*','').strip()
    ans=get_output_end(prompt+'\nA.no     B.yes')
    with open("our-methods.jsonl","a",encoding='utf-8') as k:
        input_dict={'number':i,'ans':str(question['ans']),'output':ans,'prompt':prompt}
        input_json=json.dumps(input_dict)
        k.write(input_json+'\n')

  0%|          | 0/100 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 f